In [1]:
!pip install langchain langchain-community langchain-groq chromadb \
sentence-transformers pypdf langchain-text-splitters -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/20

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

print("Uploaded:", pdf_name)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(pdf_name).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Number of Chunks:", len(chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(
    chunks,
    embeddings
)

print("Vector Database Ready!")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="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",
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
def answer_from_pdf(query):

    docs = db.as_retriever().invoke(query)

    context = "\n".join(
        d.page_content for d in docs
    )

    prompt = (
        f"Use ONLY this context to answer.\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        "If the context does not contain the answer, "
        "reply exactly: 'I don't know'."
    )

    return llm.invoke(prompt).content

In [ ]:
def adaptive_answer(question, max_tries=3):

    query = question

    for attempt in range(1, max_tries + 1):

        print(f"Attempt {attempt}: {query}")

        answer = answer_from_pdf(query)

        if "i don't know" not in answer.lower():
            return answer

        query = llm.invoke(
            f"Rephrase this search query differently: {query}"
        ).content

    return "Could not find an answer."

In [ ]:
print(
    adaptive_answer(
        "What is the main conclusion of the document?"
    )
)